In [0]:
from utils.spark_utils import get_spark
from utils.logger import get_logger
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

spark = get_spark()
logger = get_logger()
logger.info("Starting Silver Layer")

In [0]:
import json

with open("../../configs/entities.json") as f:
    config = json.load(f)

config = config['entities'][0]

In [0]:
df = spark.read.table(config["bronze_table"])

In [0]:
df.count()

In [0]:
# Incremental Processing
last_processed = spark.sql(f"""
    SELECT MAX(ingestion_time) FROM {config["silver_table"]}
""").collect()[0][0]
df = df.filter(col("ingestion_time") > last_processed)

In [0]:
df.count()

In [0]:
# expected columns check
expected_cols = ["coin", "price", "ingestion_time"]

for col_name in expected_cols:
    if col_name not in df.columns:
        df = df.withColumn(col_name, lit(None))

# drop duplicated
df_clean = df.dropDuplicates(["coin", "timestamp"])

## Data Quality Gate (MANDATORY)

In [0]:
# defensive check
count = df_clean.count()

if count == 0:
    raise Exception("No valid data for Gold layer")

In [0]:

# Before writing to Silver, validate data

from pyspark.sql.functions import col

df_valid = df_clean.filter(col("price").isNotNull())

df_invalid = df_clean.filter(col("price").isNull())

In [0]:
# Quarantine Layer
if df_invalid.count() > 0:
  print("Invalid records found. Writing to quarantine layer.")
  df_invalid.write.mode("append").saveAsTable("workspace.quarantine.crypto")
else:
  print("No invalid records found.")

In [0]:
# Idempotent Writes (Prevents Duplicate Data)
target = config["silver_table"]

if spark.catalog.tableExists(target):

    delta_table = DeltaTable.forName(spark, target)

    delta_table.alias("t").merge(
        df_clean.alias("s"),
        "t.coin = s.coin AND t.timestamp = s.timestamp"
    ).whenNotMatchedInsertAll().execute()

else:
    df_clean.write.partitionBy("coin") \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target)

In [0]:
logger.info("Silver Load Complete")